# Evaluating RAG End-to-End

You cannot improve what you do not measure. RAG systems have **two failure surfaces** — retrieval and generation — and end-to-end quality depends on both. A golden answer with wrong chunks is luck; perfect retrieval with a hallucinated answer is still a product failure.

**RAG evaluation** combines **retrieval metrics** (did the right chunks appear in top-k?) with **generation metrics** (was the answer correct, grounded, and appropriately refused?). This notebook builds a small **eval harness** on the **c1–c5** corpus: labeled queries, Recall@k and MRR, full RAG runs, substring checks, LLM-as-judge groundedness, refusal accuracy, and citation validation.

Retrieval-only depth lives in **05.09 Evaluating Retrieval Quality**. Failure taxonomy lives in **08**. This notebook wires both layers into **regression-ready** reports.

Prerequisites: **08. RAG Failure Modes and Grounding**, **07. Advanced Retrieval for RAG**, **05.09** (retrieval metrics), **02. Naive RAG Pipeline**. Next: **10. Production Patterns for RAG**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import re
from dataclasses import dataclass, field

import chromadb
import numpy as np
import matplotlib.pyplot as plt
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
REFUSAL_PHRASE = "I don't have that in the docs."

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

DOCUMENTS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI and stores notes in memory for demos. Routes live under /notes.", "source": "api-docs"},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling new revisions.", "source": "db-docs"},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Clients send Authorization: Bearer token header.", "source": "security-docs"},
    {"id": "c4", "text": "Pytest fixtures share database setup across tests. Use conftest.py for session-scoped engines.", "source": "test-docs"},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live database schema and drafts revision files.", "source": "db-docs"},
]

print("Setup OK")

---

## 1. Retrieval Eval vs Generation Eval

```
User query
    │
    ▼
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Retrieval  │ ──► │  Prompt +   │ ──► │  Generated  │
│  (top-k)    │     │  LLM        │     │  answer     │
└─────────────┘     └─────────────┘     └─────────────┘
      ▲                                         ▲
 Recall@k, MRR                          Faithfulness,
 (**05.09**, this nb)                  refusal, citations
```

| Symptom | Diagnosis | Fix layer |
|---------|-----------|----------|
| High Recall@k, bad answers | Generation / prompt | **05**, **08** |
| Low Recall@k, fluent wrong answers | Retrieval | **04**, **05**, **07** |
| Good answers, fake citations | Post-validation | **08** |
| Answers when should refuse | Abstention policy | **05**, **08** |

**Always report retrieval and generation metrics separately** before blending into a single "RAG score."

---

## 2. Golden Dataset Design

A **golden dataset** is a labeled set of test queries with expected outcomes.

### 2.1 Schema

| Field | Purpose | Example |
|-------|---------|--------|
| `query` | User question | "How do clients authenticate?" |
| `gold_ids` | Relevant chunk ids | `["c3"]` |
| `reference_answer` | Key facts expected | "JWT bearer Authorization header" |
| `must_contain` | Substrings for auto-check | `["jwt", "bearer"]` |
| `answerable` | False = should refuse | `True` / `False` |
| `query_type` | Coverage tag | `direct`, `paraphrase`, `negative` |

### 2.2 Coverage Requirements

| Type | Example | Tests |
|------|---------|-------|
| **Direct** | "Alembic upgrade command" | Basic retrieval |
| **Paraphrase** | "How do clients prove identity?" | Embedding generalization |
| **Multi-relevant** | "database schema changes" | c2 + c5 |
| **Negative / unanswerable** | "Does API use GraphQL?" | Refusal (**08**) |

Aim for **30–100** labeled queries in production; **8–12** suffices for learning demos.

In [ ]:
GOLDEN = [
    {
        "query": "How do clients authenticate to the API?",
        "gold_ids": ["c3"],
        "must_contain": ["jwt", "bearer"],
        "answerable": True,
        "query_type": "paraphrase",
    },
    {
        "query": "What command applies Alembic migrations?",
        "gold_ids": ["c2"],
        "must_contain": ["alembic", "upgrade"],
        "answerable": True,
        "query_type": "direct",
    },
    {
        "query": "Where are pytest fixtures shared across tests?",
        "gold_ids": ["c4"],
        "must_contain": ["conftest"],
        "answerable": True,
        "query_type": "direct",
    },
    {
        "query": "What does Alembic autogenerate compare?",
        "gold_ids": ["c5"],
        "must_contain": ["autogenerate", "schema"],
        "answerable": True,
        "query_type": "direct",
    },
    {
        "query": "What framework is the Notes API built with?",
        "gold_ids": ["c1"],
        "must_contain": ["fastapi"],
        "answerable": True,
        "query_type": "direct",
    },
    {
        "query": "database schema migrations and revisions",
        "gold_ids": ["c2", "c5"],
        "must_contain": ["alembic"],
        "answerable": True,
        "query_type": "multi_relevant",
    },
    {
        "query": "Does the Notes API support GraphQL?",
        "gold_ids": [],
        "must_contain": [],
        "answerable": False,
        "query_type": "negative",
    },
    {
        "query": "What is the billing refund policy?",
        "gold_ids": [],
        "must_contain": [],
        "answerable": False,
        "query_type": "negative",
    },
]

answerable = sum(1 for g in GOLDEN if g["answerable"])
print(f"Golden set: {len(GOLDEN)} queries ({answerable} answerable, {len(GOLDEN)-answerable} negative)")

---

## 3. Index Corpus and Retrieval Helpers

In [ ]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in sorted(r.data, key=lambda x: x.index)]


def fresh_collection(name: str = "rag_eval"):
    client = chromadb.Client()
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name=name, metadata={"hnsw:space": "cosine"})


collection = fresh_collection()
for d in DOCUMENTS:
    collection.add(
        ids=[d["id"]], documents=[d["text"]], embeddings=embed_texts([d["text"]]),
        metadatas=[{"source": d["source"]}],
    )
print(f"Indexed {collection.count()} chunks")

In [ ]:
def retrieve_ids(query: str, k: int = 5) -> list[str]:
    q_emb = embed_texts([query])[0]
    r = collection.query(query_embeddings=[q_emb], n_results=k, include=["distances"])
    return r["ids"][0]


def retrieve_with_docs(query: str, k: int = 3) -> tuple[list[str], str]:
    ids = retrieve_ids(query, k=k)
    got = collection.get(ids=ids, include=["documents"])
    ctx = "\n\n".join(got["documents"])
    return ids, ctx

---

## 4. Retrieval Metrics

### 4.1 Recall@k (per query)

For multi-gold queries:

$$\text{Recall@k}(q) = \frac{|\text{gold}(q) \cap \text{top-}k(q)|}{|\text{gold}(q)|}$$

For **hit rate** (any relevant in top-k): score is 1 if intersection non-empty, else 0.

### 4.2 Mean Reciprocal Rank (MRR)

$$\text{MRR} = \frac{1}{|Q|} \sum_{q} \frac{1}{\text{rank of first relevant hit}}$$

### 4.3 Precision@k

$$\text{Precision@k}(q) = \frac{|\text{gold}(q) \cap \text{top-}k(q)|}{k}$$

In [ ]:
def recall_at_k(gold_ids: list[str], retrieved: list[str], k: int) -> float:
    if not gold_ids:
        return float("nan")
    top = set(retrieved[:k])
    return len(top & set(gold_ids)) / len(gold_ids)


def hit_at_k(gold_ids: list[str], retrieved: list[str], k: int) -> float:
    if not gold_ids:
        return float("nan")
    return float(bool(set(retrieved[:k]) & set(gold_ids)))


def precision_at_k(gold_ids: list[str], retrieved: list[str], k: int) -> float:
    if not gold_ids:
        return float("nan")
    top = retrieved[:k]
    return len(set(top) & set(gold_ids)) / len(top)


def reciprocal_rank(gold_ids: list[str], retrieved: list[str]) -> float:
    if not gold_ids:
        return float("nan")
    gold = set(gold_ids)
    for rank, cid in enumerate(retrieved, start=1):
        if cid in gold:
            return 1.0 / rank
    return 0.0


K = 3
recalls, hits, mrrs = [], [], []
for item in GOLDEN:
    if not item["answerable"]:
        continue
    retrieved = retrieve_ids(item["query"], k=K)
    recalls.append(recall_at_k(item["gold_ids"], retrieved, K))
    hits.append(hit_at_k(item["gold_ids"], retrieved, K))
    mrrs.append(reciprocal_rank(item["gold_ids"], retrieved))
    print(f"[{item['query_type']:14s}] R@{K}={recalls[-1]:.2f}  MRR={mrrs[-1]:.2f}  ids={retrieved}")

print(f"\nMean Recall@{K}: {np.mean(recalls):.3f}")
print(f"Hit rate@{K}:     {np.mean(hits):.3f}")
print(f"MRR:              {np.mean(mrrs):.3f}")

---

## 5. Generation Metrics

| Metric | How to compute | When |
|--------|----------------|------|
| **must_contain** | All substrings in answer (case-insensitive) | Cheap regression |
| **Refusal accuracy** | Unanswerable → contains refusal phrase | Negative set (**08**) |
| **Citation validity** | Cited ids ⊆ retrieved ids | **08** allow-list |
| **LLM-as-judge** | Second model scores groundedness 1–5 | Paraphrase-tolerant |
| **Exact match** | Normalized string == reference | Rare for open answers |

In [ ]:
SOURCE_PATTERN = re.compile(r"Sources:\s*([\w,\s]+)", re.IGNORECASE)
CHUNK_ID_PATTERN = re.compile(r"c\d+")


def parse_cited_ids(answer: str) -> list[str]:
    m = SOURCE_PATTERN.search(answer)
    return CHUNK_ID_PATTERN.findall(m.group(1)) if m else []


def must_contain_pass(answer: str, terms: list[str]) -> bool:
    lower = answer.lower()
    return all(t.lower() in lower for t in terms)


def refusal_detected(answer: str) -> bool:
    return REFUSAL_PHRASE.lower() in answer.lower() or "don't know" in answer.lower()


def citation_valid(answer: str, retrieved_ids: set[str]) -> bool:
    cited = parse_cited_ids(answer)
    if not cited:
        return True  # no citations claimed
    return all(c in retrieved_ids for c in cited)

---

## 6. Full RAG Pipeline for Eval

In [ ]:
RAG_SYSTEM = f"""Answer ONLY from Context. If insufficient, say exactly: {REFUSAL_PHRASE}
End with: Sources: <comma-separated chunk ids>"""


def rag_answer(query: str, k: int = 3) -> tuple[str, list[str]]:
    ids, ctx = retrieve_with_docs(query, k=k)
    labeled_ctx = "\n\n".join(
        f"[{cid}] {doc}" for cid, doc in zip(ids, ctx.split("\n\n"))
    )
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"Context:\n{labeled_ctx}\n\nQuestion: {query}"},
        ],
    )
    return r.choices[0].message.content, ids

---

## 7. Run Generation Checks on Golden Set

Replace API key to run live. Each row logs retrieval + generation outcomes.

In [ ]:
@dataclass
class EvalRow:
    query: str
    query_type: str
    answerable: bool
    retrieved_ids: list[str] = field(default_factory=list)
    recall_at_k: float = 0.0
    answer: str = ""
    must_contain_ok: bool = False
    refusal_ok: bool = False
    citation_ok: bool = True


def eval_one(item: dict, k: int = 3) -> EvalRow:
    row = EvalRow(query=item["query"], query_type=item["query_type"], answerable=item["answerable"])
    row.retrieved_ids = retrieve_ids(item["query"], k=k)
    if item["answerable"]:
        row.recall_at_k = recall_at_k(item["gold_ids"], row.retrieved_ids, k)
    row.answer, gen_ids = rag_answer(item["query"], k=k)
    row.must_contain_ok = must_contain_pass(row.answer, item["must_contain"]) if item["answerable"] else True
    row.refusal_ok = (not item["answerable"] and refusal_detected(row.answer)) or item["answerable"]
    row.citation_ok = citation_valid(row.answer, set(row.retrieved_ids))
    return row


# Run on first 2 answerable + 1 negative for demo (expand to full GOLDEN in practice)
DEMO_ITEMS = [GOLDEN[0], GOLDEN[1], GOLDEN[6]]
demo_rows = [eval_one(item) for item in DEMO_ITEMS]
for row in demo_rows:
    print(f"\nQ: {row.query[:50]}...")
    print(f"  recall={row.recall_at_k:.2f}  must_contain={row.must_contain_ok}  refusal_ok={row.refusal_ok}  cite_ok={row.citation_ok}")
    print(f"  answer: {row.answer[:120]}...")

---

## 8. LLM-as-Judge Groundedness

A second LLM call scores whether **every claim** in the answer is supported by context.

| Pros | Cons |
|------|------|
| Handles paraphrase | Judge has its own bias |
| Faster than human review | Costs tokens |
| Scales to large eval sets | Needs fixed rubric |

Use `temperature=0` and structured JSON output.

In [ ]:
def judge_groundedness(context: str, answer: str) -> dict:
    rubric = (
        "Score 1-5: is EVERY factual claim in the answer supported by the context? "
        "5 = fully supported, 1 = mostly unsupported. "
        "Reply JSON: {\"score\": int, \"rationale\": string}"
    )
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": rubric},
            {"role": "user", "content": f"Context:\n{context}\n\nAnswer:\n{answer}"},
        ],
    )
    return json.loads(r.choices[0].message.content)


ctx = DOCUMENTS[2]["text"]
good = "Clients send JWT bearer tokens in the Authorization header."
bad = "Clients use OAuth2 and SAML for authentication."
print("Good answer judge:", judge_groundedness(ctx, good))
print("Bad answer judge: ", judge_groundedness(ctx, bad))

---

## 9. Aggregate Eval Report

Roll per-query rows into a **regression report** suitable for CI thresholds.

In [ ]:
@dataclass
class RAGEvalReport:
    n_queries: int
    mean_recall_at_k: float
    mrr: float
    must_contain_rate: float
    refusal_accuracy: float
    citation_valid_rate: float
    k: int
    embed_model: str
    chat_model: str


def run_full_eval(golden: list[dict], k: int = 3) -> tuple[list[EvalRow], RAGEvalReport]:
    rows = [eval_one(item, k=k) for item in golden]
    answerable = [r for r in rows if r.answerable]
    negative = [r for r in rows if not r.answerable]
    mrr_vals = [reciprocal_rank(g["gold_ids"], r.retrieved_ids) for g, r in zip(
        [x for x in golden if x["answerable"]], answerable
    )]
    report = RAGEvalReport(
        n_queries=len(rows),
        mean_recall_at_k=float(np.mean([r.recall_at_k for r in answerable])) if answerable else 0.0,
        mrr=float(np.mean(mrr_vals)) if mrr_vals else 0.0,
        must_contain_rate=float(np.mean([r.must_contain_ok for r in answerable])) if answerable else 0.0,
        refusal_accuracy=float(np.mean([r.refusal_ok for r in negative])) if negative else 1.0,
        citation_valid_rate=float(np.mean([r.citation_ok for r in rows])),
        k=k,
        embed_model=EMBED_MODEL,
        chat_model=CHAT_MODEL,
    )
    return rows, report


# Full eval — requires API key; comment out if offline
# rows, report = run_full_eval(GOLDEN, k=3)
# print(json.dumps(report.__dict__, indent=2))

---

## 10. Miss Analysis

When Recall@k fails, categorize **why** — guides the next fix (**08**).

| Miss type | Signal | Next step |
|-----------|--------|----------|
| **Paraphrase** | Gold id rank 8–15 | Hybrid / multi-query (**07**) |
| **Chunking** | Answer split across chunks | **05.04** |
| **Wrong corpus** | Irrelevant high scores | Metadata filter |
| **Stale index** | Old text retrieved | Re-ingest (**04**) |
| **Generation** | Recall OK, wrong answer | **05**, **08** |

In [ ]:
def miss_analysis(item: dict, retrieved: list[str]) -> str:
    if not item["answerable"]:
        return "negative_query"
    gold = set(item["gold_ids"])
    if gold & set(retrieved):
        return "hit"
    # check if gold in top-10 for partial miss
    top10 = retrieve_ids(item["query"], k=10)
    if gold & set(top10):
        return "rank_miss (in top-10, not top-k)"
    return "retrieval_miss (not in top-10)"


for item in GOLDEN:
    if not item["answerable"]:
        continue
    ret = retrieve_ids(item["query"], k=3)
    print(f"{miss_analysis(item, ret):30s}  {item['query'][:45]}")

---

## 11. Regression Testing in CI

Run eval on every PR that touches prompts, ingest, or retrieval:

```bash
pytest tests/rag_eval.py --min-recall 0.85 --min-refusal 0.9
```

| Practice | Benefit |
|----------|--------|
| Pin golden set in git | Reproducible baselines |
| Fail on regression vs `main` | Catch prompt/index drift |
| Cache query embeddings | Faster CI |
| Subset judge calls | Cost control |

---

## 12. Cost-Aware Eval Strategy

| Tier | When | What runs |
|------|------|----------|
| **PR** | Every commit | Recall@k only (no LLM) |
| **Nightly** | Scheduled | Full RAG + must_contain |
| **Weekly** | Deep dive | LLM-as-judge on full set |
| **Human** | Sample 50/week | Edge cases for new golden rows |

Cache `embed(query)` for golden questions — identical across runs unless embed model changes.

---

## 13. Reporting Dashboard

| Chart | Shows |
|-------|-------|
| Recall@k over time | Index / embed regressions |
| Refusal accuracy | Abstention policy health |
| Failure mode pie | Labels from **08** |
| Latency p95 | Production SLA (**10**) |
| Cost per 1k queries | Embed + chat tokens |

In [ ]:
# Illustrative metric trends (synthetic — replace with logged production data)
weeks = ["W1", "W2", "W3", "W4"]
recall_trend = [0.82, 0.88, 0.86, 0.91]
refusal_trend = [0.75, 0.85, 0.90, 0.92]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(weeks, recall_trend, marker="o", label="Recall@3")
ax[0].set_ylim(0.7, 1.0)
ax[0].set_title("Retrieval quality")
ax[0].grid(True, alpha=0.3)
ax[1].plot(weeks, refusal_trend, marker="s", color="green", label="Refusal accuracy")
ax[1].set_ylim(0.7, 1.0)
ax[1].set_title("Negative-set refusal")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 14. Human Evaluation Loop

Automated metrics miss nuance. Weekly sample **production queries** and label:

1. Retrieval: relevant / partial / miss
2. Answer: correct / partial / wrong
3. Grounding: supported / unsupported / mixed

Feed failures into `GOLDEN` — especially edge cases and new product areas.

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Single end-to-end score | Can't localize bugs | Split retrieval vs generation |
| No negative examples | Hallucination on OOD | Unanswerable rows |
| Tiny golden set (n=3) | Overfit to prompts | Expand to 30+ |
| LLM judge only | Expensive, biased | Start with must_contain |
| Skip citation check | Fake sources ship | **08** allow-list |
| Eval once at launch | Regressions undetected | CI regression |

---

## 16. Summary

End-to-end RAG eval combines **Recall@k / MRR** on chunk ids with **generation checks**: must_contain substrings, refusal accuracy on negatives, citation validation, and optional **LLM-as-judge** groundedness.

The harness on **c1–c5** — `GOLDEN`, `eval_one`, `run_full_eval`, and `RAGEvalReport` — is a template. Expand the golden set as you add real docs; run cheap retrieval metrics on every PR and full generation eval nightly.

Measure separately, fix the bottleneck, re-run. Notebook **10** operationalizes logging, versioning, and deployment around this eval loop.

Back: **08. RAG Failure Modes and Grounding**. Next: **10. Production Patterns for RAG**.